# D243 — Parquet Tables in Apache Hive

This short lab creates a managed Parquet table, inserts rows with Hive SQL, examines the generated files in HDFS, and registers copied Parquet files as an external table using `LOCATION`.

**Environment:** Hive 4.0.1, Hadoop 3.3.6, HDFS, YARN/MapReduce, and Beeline.

## 1. What is Parquet?

Apache Parquet is a **columnar binary file format**. Values from the same column are stored together, allowing analytical queries to read only the columns they need. Parquet also stores schema and statistics in its file metadata and usually compresses repeated values efficiently.

Parquet is well suited to analytical tables. Unlike CSV or JSON, it is not human-readable and should not be created with `tee`. In this lesson, Hive writes the Parquet files for us.

## 2. Service spot-check

In [ ]:
%%bash
jps
ss -lnt | grep -E ':(9083|10000)\b' || true
hdfs dfsadmin -report | grep -E 'Live datanodes|Name:'

Expected: one live DataNode, Hive metastore on port `9083`, and HiveServer2 on port `10000`.

## 3. Create the lab database

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/default' -n "$USER" --silent=true -e "
CREATE DATABASE IF NOT EXISTS hive_parquet
COMMENT 'Parquet examples for the D243 lab';
DESCRIBE DATABASE EXTENDED hive_parquet;
"

## 4. Create a managed Parquet table

A normal `CREATE TABLE` creates a managed table. `STORED AS PARQUET` tells Hive to serialize its rows as Parquet files in the database's warehouse directory.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_parquet' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS sales_parquet;
CREATE TABLE sales_parquet (
  sale_id INT,
  product STRING,
  category STRING,
  quantity INT,
  unit_price DECIMAL(10,2),
  sale_date DATE
)
STORED AS PARQUET
TBLPROPERTIES ('parquet.compression'='SNAPPY');
DESCRIBE FORMATTED sales_parquet;
"

In `DESCRIBE FORMATTED`, look for:

- `Table Type: MANAGED_TABLE`;
- a warehouse `Location`;
- `MapredParquetInputFormat` and `MapredParquetOutputFormat`;
- the Snappy compression table property.

## 5. Insert rows

`INSERT INTO` appends rows. Multiple value tuples can be inserted in one statement. Hive converts the typed values into Parquet's binary columnar representation.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_parquet' -n "$USER" --silent=true -e "
INSERT INTO sales_parquet VALUES
  (1, 'Laptop',   'Electronics', 1, 65000.00, DATE '2026-08-18'),
  (2, 'Mouse',    'Electronics', 2,   850.00, DATE '2026-08-18'),
  (3, 'Chair',    'Furniture',   1,  7500.00, DATE '2026-08-19');

INSERT INTO sales_parquet VALUES
  (4, 'Desk',     'Furniture',   1, 12000.00, DATE '2026-08-19'),
  (5, 'Notebook', 'Stationery',  5,   120.00, DATE '2026-08-20');

SELECT * FROM sales_parquet ORDER BY sale_id;
"

Because two `INSERT` statements were used, Hive may create more than one Parquet data file. Many small inserts can therefore produce many small files; production pipelines normally write data in larger batches.

### Query selected columns

A columnar format can avoid reading unused columns. This query needs only `category`, `quantity`, and `unit_price` from the six-column table.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_parquet' -n "$USER" --silent=true -e "
SELECT
  category,
  SUM(quantity) AS units,
  SUM(quantity * unit_price) AS revenue
FROM sales_parquet
GROUP BY category
ORDER BY category;
"

Expected revenue: Electronics 66700, Furniture 19500, and Stationery 600.

## 6. Inspect the generated Parquet files in HDFS

The managed table's default location is under the Hive warehouse. `hdfs dfs -ls` shows the files, but `hdfs dfs -cat` is not useful because Parquet is binary.

In [ ]:
%%bash
MANAGED_PATH=/user/hive/warehouse/hive_parquet.db/sales_parquet
echo "Managed-table path: $MANAGED_PATH"
hdfs dfs -ls "$MANAGED_PATH"
hdfs dfs -du -h "$MANAGED_PATH"

If your warehouse path is customized, copy the exact `Location` printed by `DESCRIBE FORMATTED sales_parquet` and update `MANAGED_PATH`.

## 7. Prepare a Parquet location for an external table

An external Parquet table expects **existing Parquet files**, not CSV text. We copy the files Hive just generated into a dedicated external-data directory. The source and destination stay in HDFS, so no local conversion is required.

The cleanup is intentionally restricted to this lab's exact destination.

In [ ]:
%%bash
MANAGED_PATH=/user/hive/warehouse/hive_parquet.db/sales_parquet
EXTERNAL_PATH=/user/hive/external/d243_sales_parquet
hdfs dfs -rm -r -f "$EXTERNAL_PATH"
hdfs dfs -mkdir -p "$EXTERNAL_PATH"
hdfs dfs -cp "$MANAGED_PATH"/* "$EXTERNAL_PATH"/
echo "External Parquet location: $EXTERNAL_PATH"
hdfs dfs -ls "$EXTERNAL_PATH"

## 8. Create an external Parquet table from `LOCATION`

The external table must declare a schema compatible with the schema stored in the Parquet files. Hive registers metadata pointing at the directory; it does not copy the files during `CREATE EXTERNAL TABLE`.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_parquet' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS sales_parquet_external;
CREATE EXTERNAL TABLE sales_parquet_external (
  sale_id INT,
  product STRING,
  category STRING,
  quantity INT,
  unit_price DECIMAL(10,2),
  sale_date DATE
)
STORED AS PARQUET
LOCATION '/user/hive/external/d243_sales_parquet';
SELECT * FROM sales_parquet_external ORDER BY sale_id;
DESCRIBE FORMATTED sales_parquet_external;
"

Look for `EXTERNAL_TABLE`, the explicit location, and Parquet input/output formats. The external table returns the same five rows because its directory contains copies of the managed table's Parquet files.

## 9. Prove the external-data lifecycle

Dropping an external table removes its catalog entry but leaves the files at its `LOCATION`.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_parquet' -n "$USER" --silent=true -e "
DROP TABLE sales_parquet_external;
SHOW TABLES LIKE 'sales_parquet_external';
"
echo 'The external Parquet files still exist:'
hdfs dfs -ls /user/hive/external/d243_sales_parquet

Recreate the table over the unchanged location.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_parquet' -n "$USER" --silent=true -e "
CREATE EXTERNAL TABLE sales_parquet_external (
  sale_id INT, product STRING, category STRING, quantity INT,
  unit_price DECIMAL(10,2), sale_date DATE
)
STORED AS PARQUET
LOCATION '/user/hive/external/d243_sales_parquet';
SELECT COUNT(*) AS restored_rows FROM sales_parquet_external;
"

## 10. Important Parquet notes

- Parquet files contain schema information, but Hive still needs a catalog table definition.
- Keep Hive column names, order, and types compatible with the physical Parquet schema.
- Do not mix CSV, JSON, and Parquet files in the same table location.
- Avoid many tiny inserts because they can create many small files.
- Parquet is binary; query it with Hive or a Parquet-aware tool rather than `cat`.
- Dropping a managed table normally deletes its files. Dropping an external table normally preserves files at its location.

## Optional cleanup

Keep the objects for later lessons, or remove the catalog entries:

```sql
USE hive_parquet;
DROP TABLE IF EXISTS sales_parquet_external;
DROP TABLE IF EXISTS sales_parquet;
```

The managed table's files are removed when it is dropped. The external files remain at `/user/hive/external/d243_sales_parquet` until that exact HDFS directory is explicitly removed.